# French Property Intelligence
## 03 — Baseline Models

### Objective

This notebook establishes reproducible baseline performance for the property valuation system.

Two specialized valuation tasks are modeled independently:

1. houses (`Maison`);
2. apartments (`Appartement`).

The frozen train and validation datasets produced in Notebook 02 are reused without modifying their observations.

The initial experiments are:

1. a naive constant-price baseline;
2. a regularized linear Ridge regression baseline.

These baselines provide reference performance against which more expressive models, including LightGBM, can later be evaluated.

### Evaluation strategy

Model selection is based primarily on **Mean Absolute Error (MAE)** because the target contains a strongly right-skewed price distribution and the business interpretation of absolute valuation error is straightforward.

Secondary metrics are:

- RMSE, to expose sensitivity to large errors;
- R², to measure explained variance.

The test datasets remain untouched during model development.

### Production constraint

Preprocessing and prediction logic must eventually be serializable as a single reproducible model pipeline suitable for:

- MLflow experiment tracking;
- storage as a model artifact in AWS S3;
- loading by the production FastAPI service;
- prediction from inputs supplied by the Streamlit application.

Consequently, preprocessing learned from data will be fitted using training data only and incorporated into model pipelines whenever possible.

In [50]:
# PURPOSE:
# Import the libraries required for baseline modeling and define
# reproducibility and project paths.
#
# We intentionally do not import LightGBM yet. This notebook first
# establishes simple reference models against which advanced models
# can later be compared.

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Import preprocessing and modeling components required to build
# a single serializable Ridge pipeline.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone


RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data:", DATA_PROCESSED_DIR)

Project root: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\6_french-property-intelligence
Processed data: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\6_french-property-intelligence\data\processed


In [51]:
# PURPOSE:
# Load the frozen training and validation populations created in
# Notebook 02.
#
# Test datasets are deliberately not loaded during model development
# to protect the final evaluation from model-selection decisions.

house_train = pd.read_parquet(
    DATA_PROCESSED_DIR / "houses_train.parquet"
)

house_val = pd.read_parquet(
    DATA_PROCESSED_DIR / "houses_validation.parquet"
)

apartment_train = pd.read_parquet(
    DATA_PROCESSED_DIR / "apartments_train.parquet"
)

apartment_val = pd.read_parquet(
    DATA_PROCESSED_DIR / "apartments_validation.parquet"
)


print("HOUSES")
print("Train:     ", house_train.shape)
print("Validation:", house_val.shape)

print("\nAPARTMENTS")
print("Train:     ", apartment_train.shape)
print("Validation:", apartment_val.shape)

HOUSES
Train:      (1630944, 11)
Validation: (349176, 11)

APARTMENTS
Train:      (1263001, 11)
Validation: (269977, 11)


In [52]:
# PURPOSE:
# Define the initial minimal production-compatible feature contract.
#
# Fine geographic location is represented by latitude and longitude.
# Postal code and department are retained in the datasets for possible
# controlled feature-ablation experiments, but are not initial predictors.

TARGET = "prix"

NUMERIC_FEATURES = [
    "surface_habitable",
    "n_pieces",
    "latitude",
    "longitude",
]

BOOLEAN_FEATURES = [
    "vefa",
]

MODEL_FEATURES = (
    NUMERIC_FEATURES
    + BOOLEAN_FEATURES
)

AUDIT_ONLY = [
    "date_transaction",
    "type_batiment",
    "code_postal",
    "departement",
    "id_parcelle_cadastre",
]

print("Model features:", MODEL_FEATURES)
print("Audit / experimental only:", AUDIT_ONLY)

Model features: ['surface_habitable', 'n_pieces', 'latitude', 'longitude', 'vefa']
Audit / experimental only: ['date_transaction', 'type_batiment', 'code_postal', 'departement', 'id_parcelle_cadastre']


In [53]:
# PURPOSE:
# Verify that the frozen datasets satisfy the expected modeling
# contract before fitting any baseline.
#
# This catches accidental schema changes early and will later inspire
# similar validation inside the production API.

datasets = {
    "house_train": house_train,
    "house_validation": house_val,
    "apartment_train": apartment_train,
    "apartment_validation": apartment_val,
}

required_columns = set(
    MODEL_FEATURES
    + [TARGET]
    + AUDIT_ONLY
)

for name, df in datasets.items():

    missing_columns = required_columns - set(df.columns)

    print(
        f"{name:<25} "
        f"rows={len(df):>10,}  "
        f"missing_required_columns={len(missing_columns)}"
    )

    assert not missing_columns, (
        f"{name} is missing: {missing_columns}"
    )

print("\n✓ Modeling schemas validated.")

house_train               rows= 1,630,944  missing_required_columns=0
house_validation          rows=   349,176  missing_required_columns=0
apartment_train           rows= 1,263,001  missing_required_columns=0
apartment_validation      rows=   269,977  missing_required_columns=0

✓ Modeling schemas validated.


## 2. Evaluation framework

All candidate models are evaluated using the same validation observations and the same metrics:

- **MAE** — primary model-selection metric;
- **RMSE** — secondary metric emphasizing large prediction errors;
- **R²** — secondary measure of explained variance.

MAE is the primary metric because property prices are strongly right-skewed and the absolute prediction error has a direct business interpretation in euros.

Model-selection decisions are made using validation data only. The test datasets remain untouched until the final modeling approach has been selected.

In [54]:
# PURPOSE:
# Define a single evaluation function that will be reused for every
# candidate model.
#
# Using the same function prevents metric inconsistencies between
# the naive baseline, Ridge, LightGBM, and later experiments.

def evaluate_regression(y_true, y_pred, model_name, property_type):
    """
    Calculate the project's standard regression metrics.

    MAE is the primary model-selection metric.
    RMSE and R² are secondary diagnostic metrics.
    """

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    return {
        "property_type": property_type,
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }

## 3. Naive median-price baseline

Before fitting a machine-learning model, a simple reference prediction is established independently for houses and apartments.

For each property type, the baseline always predicts the **median transaction price observed in its training dataset**.

The validation target is never used to calculate this value.

This provides a minimum benchmark: subsequent models should demonstrate meaningful improvement over a prediction that ignores all property and geographic characteristics.

In [55]:
# PURPOSE:
# Calculate the naive prediction from TRAINING DATA ONLY.
#
# Every validation property receives the corresponding training median
# price. This establishes the minimum benchmark that ML models must beat.

house_median_price = house_train[TARGET].median()
apartment_median_price = apartment_train[TARGET].median()

print(f"House training median:     €{house_median_price:,.0f}")
print(f"Apartment training median: €{apartment_median_price:,.0f}")


# Create constant predictions for the validation populations.
house_naive_pred = np.full(
    len(house_val),
    house_median_price
)

apartment_naive_pred = np.full(
    len(apartment_val),
    apartment_median_price
)

House training median:     €195,600
Apartment training median: €166,000


In [56]:
# PURPOSE:
# Evaluate the naive house and apartment baselines using exactly the
# same metrics that will later evaluate Ridge and LightGBM.

baseline_results = []

baseline_results.append(
    evaluate_regression(
        y_true=house_val[TARGET],
        y_pred=house_naive_pred,
        model_name="Naive median",
        property_type="House",
    )
)

baseline_results.append(
    evaluate_regression(
        y_true=apartment_val[TARGET],
        y_pred=apartment_naive_pred,
        model_name="Naive median",
        property_type="Apartment",
    )
)

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,property_type,model,MAE,RMSE,R2
0,House,Naive median,133558.730270,256470.675483,-0.046472
1,Apartment,Naive median,122966.483477,255155.367456,-0.059030


In [57]:
# PURPOSE:
# Present the baseline errors in an interpretable euro format.
#
# These values will later make it easy to quantify how much each
# machine-learning model improves over the naive reference.

for result in baseline_results:
    print(
        f"{result['property_type']:<10} | "
        f"{result['model']:<15} | "
        f"MAE: €{result['MAE']:,.0f} | "
        f"RMSE: €{result['RMSE']:,.0f} | "
        f"R²: {result['R2']:.4f}"
    )

House      | Naive median    | MAE: €133,559 | RMSE: €256,471 | R²: -0.0465
Apartment  | Naive median    | MAE: €122,966 | RMSE: €255,155 | R²: -0.0590


## 4. Ridge regression baseline

Ridge regression provides the first genuine machine-learning baseline.

It is useful because it introduces property and geographic characteristics while remaining relatively simple and interpretable. Its performance establishes whether the available predictors contain substantial valuation signal before introducing a nonlinear tree-based model.

### Preprocessing

The Ridge model requires numerical inputs. Preprocessing is therefore incorporated into a scikit-learn `Pipeline`:

- numeric features:
  - missing room counts are imputed using the training median;
  - numeric variables are standardized;
- boolean features are converted to numeric values;
- categorical geographic variables are one-hot encoded;
- previously unseen categories are ignored safely at prediction time.

All learned preprocessing parameters are fitted on training data only.

Packaging preprocessing and regression into one pipeline is also compatible with the intended production architecture: the resulting object can later be serialized and used to transform raw API features and generate a prediction consistently.

In [58]:
# PURPOSE:
# Build the initial Ridge preprocessing pipeline.
#
# Numeric variables are imputed where necessary and standardized.
# The boolean VEFA indicator is passed directly to the estimator.
#
# All learned transformations remain inside the serializable pipeline.


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            NUMERIC_FEATURES,
        ),
        (
            "boolean",
            "passthrough",
            BOOLEAN_FEATURES,
        ),
    ]
)


ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]
)


house_ridge = clone(ridge_pipeline)
apartment_ridge = clone(ridge_pipeline)

In [59]:
# PURPOSE:
# Construct feature matrices using the revised minimal feature contract.

X_house_train = house_train[MODEL_FEATURES]
y_house_train = house_train[TARGET]

X_house_val = house_val[MODEL_FEATURES]
y_house_val = house_val[TARGET]

X_apartment_train = apartment_train[MODEL_FEATURES]
y_apartment_train = apartment_train[TARGET]

X_apartment_val = apartment_val[MODEL_FEATURES]
y_apartment_val = apartment_val[TARGET]

print("House train:", X_house_train.shape)
print("House validation:", X_house_val.shape)

print("\nApartment train:", X_apartment_train.shape)
print("Apartment validation:", X_apartment_val.shape)

House train: (1630944, 5)
House validation: (349176, 5)

Apartment train: (1263001, 5)
Apartment validation: (269977, 5)


In [60]:
# PURPOSE:
# Fit the Ridge baseline for HOUSES using training data only.
#
# The pipeline learns both preprocessing and regression parameters.
# Keeping preprocessing inside the pipeline is important for later
# serialization, MLflow tracking, S3 storage, and FastAPI inference.

house_ridge.fit(
    X_house_train,
    y_house_train
)

print("✓ House Ridge fitted.")

✓ House Ridge fitted.


In [61]:
# PURPOSE:
# Evaluate the House Ridge model on the frozen validation population
# using exactly the same metrics as the naive baseline.

house_ridge_pred = house_ridge.predict(X_house_val)

house_ridge_result = evaluate_regression(
    y_true=y_house_val,
    y_pred=house_ridge_pred,
    model_name="Ridge",
    property_type="House",
)

print(
    f"House | Ridge | "
    f"MAE: €{house_ridge_result['MAE']:,.0f} | "
    f"RMSE: €{house_ridge_result['RMSE']:,.0f} | "
    f"R²: {house_ridge_result['R2']:.4f}"
)

House | Ridge | MAE: €124,039 | RMSE: €223,815 | R²: 0.2031


In [62]:
# PURPOSE:
# Quantify whether Ridge provides meaningful predictive value compared
# with simply predicting the median house price.

house_naive_mae = baseline_results_df.loc[
    baseline_results_df["property_type"] == "House",
    "MAE"
].iloc[0]

house_ridge_mae = house_ridge_result["MAE"]

mae_gain_euros = house_naive_mae - house_ridge_mae

mae_gain_pct = (
    mae_gain_euros / house_naive_mae
) * 100

print(f"Naive MAE:          €{house_naive_mae:,.0f}")
print(f"Ridge MAE:          €{house_ridge_mae:,.0f}")
print(f"MAE reduction:      €{mae_gain_euros:,.0f}")
print(f"Relative reduction: {mae_gain_pct:.2f}%")

Naive MAE:          €133,559
Ridge MAE:          €124,039
MAE reduction:      €9,519
Relative reduction: 7.13%


### House Ridge result

The Ridge baseline improves validation MAE from approximately **€133,559** for the naive median predictor to **€124,039**, corresponding to a **7.13% reduction in MAE** (approximately **€9,519**).

RMSE also decreases from approximately **€256,471** to **€223,815**, while validation R² increases from **-0.0465** for the naive baseline to **0.2031** for Ridge.

These results demonstrate that the selected property characteristics and geographic coordinates contain meaningful predictive information beyond a constant median-price prediction. However, the remaining MAE and RMSE are still substantial, and an R² of **0.2031** indicates that the linear Ridge model captures only part of the observed variation in house transaction prices.

Ridge therefore provides a useful linear benchmark, while motivating the subsequent evaluation of nonlinear models under the same frozen train/validation partitions, feature contract, and evaluation metrics.

In [63]:
# PURPOSE:
# Fit the equivalent Ridge baseline for APARTMENTS.
#
# The apartment pipeline is fitted independently so its imputation,
# scaling parameters, and regression coefficients are learned only
# from apartment training transactions.

apartment_ridge.fit(
    X_apartment_train,
    y_apartment_train
)

print("✓ Apartment Ridge fitted.")

✓ Apartment Ridge fitted.


In [64]:
# PURPOSE:
# Evaluate Apartment Ridge on the frozen apartment validation set
# using exactly the same metrics as all previous experiments.

apartment_ridge_pred = apartment_ridge.predict(
    X_apartment_val
)

apartment_ridge_result = evaluate_regression(
    y_true=y_apartment_val,
    y_pred=apartment_ridge_pred,
    model_name="Ridge",
    property_type="Apartment",
)

print(
    f"Apartment | Ridge | "
    f"MAE: €{apartment_ridge_result['MAE']:,.0f} | "
    f"RMSE: €{apartment_ridge_result['RMSE']:,.0f} | "
    f"R²: {apartment_ridge_result['R2']:.4f}"
)

Apartment | Ridge | MAE: €122,592 | RMSE: €212,375 | R²: 0.2663


In [65]:
# PURPOSE:
# Quantify the Apartment Ridge improvement relative to the
# apartment naive median-price benchmark.

apartment_naive_mae = baseline_results_df.loc[
    baseline_results_df["property_type"] == "Apartment",
    "MAE"
].iloc[0]

apartment_ridge_mae = apartment_ridge_result["MAE"]

mae_gain_euros = (
    apartment_naive_mae
    - apartment_ridge_mae
)

mae_gain_pct = (
    mae_gain_euros
    / apartment_naive_mae
    * 100
)

print(f"Naive MAE:          €{apartment_naive_mae:,.0f}")
print(f"Ridge MAE:          €{apartment_ridge_mae:,.0f}")
print(f"MAE reduction:      €{mae_gain_euros:,.0f}")
print(f"Relative reduction: {mae_gain_pct:.2f}%")

Naive MAE:          €122,966
Ridge MAE:          €122,592
MAE reduction:      €374
Relative reduction: 0.30%


### Apartment Ridge result

The Apartment Ridge baseline achieves a validation MAE of approximately **€122,592**, compared with **€122,966** for the naive median predictor. This corresponds to an MAE reduction of only approximately **€374**, or **0.30%**.

The secondary metrics improve more substantially: RMSE decreases from approximately **€255,155** for the naive baseline to **€212,375** for Ridge, while validation R² increases from **-0.0590** to **0.2663**.

This difference between metrics is informative. Ridge reduces some larger prediction errors enough to improve RMSE and R², but provides only a marginal improvement in the project's primary metric, MAE. Under the predefined model-selection criterion, the linear model therefore adds limited value over the naive median predictor for typical apartment valuation error.

Together with the house results, these findings establish Ridge as a useful linear benchmark but motivate evaluating a nonlinear model capable of representing more complex relationships between property characteristics, geographic location, and transaction price.

In [66]:
# PURPOSE:
# Consolidate all baseline experiments into one comparison table.
#
# This table becomes the reference against which the advanced nonlinear
# models will be evaluated in the next modeling phase.

baseline_comparison = pd.DataFrame([
    baseline_results[0],
    house_ridge_result,
    baseline_results[1],
    apartment_ridge_result,
])

baseline_comparison["MAE"] = baseline_comparison["MAE"].round(0)
baseline_comparison["RMSE"] = baseline_comparison["RMSE"].round(0)
baseline_comparison["R2"] = baseline_comparison["R2"].round(4)

baseline_comparison

,property_type,model,MAE,RMSE,R2
0,House,Naive median,133559.0,256471.0,-0.0465
1,House,Ridge,124039.0,223815.0,0.2031
2,Apartment,Naive median,122966.0,255155.0,-0.0590
3,Apartment,Ridge,122592.0,212375.0,0.2663


In [67]:
# PURPOSE:
# Persist baseline validation metrics so later notebooks can compare
# advanced models against exactly the same reference results.

OUTPUT_METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)

baseline_comparison.to_csv(
    OUTPUT_METRICS_DIR / "baseline_model_metrics.csv",
    index=False
)

print("✓ Baseline metrics saved.")

✓ Baseline metrics saved.


## Baseline modeling conclusion

The baseline experiments show that the selected property characteristics and geographic coordinates contain predictive information, but that a simple linear model remains insufficient for the intended valuation task.

For **houses**, Ridge reduces validation MAE from approximately **€133,559** for the naive median benchmark to **€124,039**, corresponding to a **7.13% reduction in MAE**. RMSE also decreases from approximately **€256,471** to **€223,815**, while R² increases from **-0.0465** to **0.2031**.

For **apartments**, Ridge reduces validation MAE only marginally, from approximately **€122,966** to **€122,592**, corresponding to a **0.30% reduction in MAE**. However, RMSE decreases more substantially from approximately **€255,155** to **€212,375**, while R² increases from **-0.0590** to **0.2663**.

Because **MAE is the predefined primary model-selection metric**, Ridge currently provides the strongest baseline reference for both property types:

- **House reference MAE:** €124,039 (Ridge)
- **Apartment reference MAE:** €122,592 (Ridge)

The relatively limited MAE performance, particularly for apartments, suggests that linear relationships alone do not adequately represent the complexity of residential property prices. Important relationships between surface, location, property characteristics, and transaction value are likely to be nonlinear and may involve interactions that Ridge cannot represent directly.

The next modeling phase will therefore evaluate a nonlinear gradient-boosted tree model using the **same frozen training and validation populations**, the **same initial feature contract**, and the **same evaluation metrics**. This controlled comparison will help determine whether additional model capacity provides a meaningful improvement over the established linear baselines.

The held-out test populations remain untouched and will only be used after model selection is complete.